## AI4Climate ML tutorial - Inference and Visualisation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered
In the previous notebooks we have explored and prepared a dataset for training a machine learning model, then we have trained a few different algorithms using this data. The next step is to use the trained model,  both to evaluate and understand how well it has learned the relationship in the data we want it to learn, but also then aplying to the intended use of the data. For example if we have trained a global climate model, we want to use the trained model for experiments around climate change and climate variability, for example.  In this notebook we will look at runnjing inference with the model and visualising the results.


### Prerequisites 
- Same as previouis notebooks
- Have completed model training pipeline notebook


### Learning outcomes from completing the notebook
* Load a saved model
* Make predictions with the model
* Visualise the results

## Tutorial 
a balance of explanation and activity



In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import pandas

In [1]:
import iris
import cartopy.crs

In [2]:
import matplotlib.pyplot

In [3]:
import mlflow

In [3]:
import sklearn
import sklearn.preprocessing
import sklearn.tree

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding


In [6]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, very cold winter',
  'Dw

In [7]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [8]:
current_platform = tutorial_config['platform']

In [9]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones')

In [10]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready')

In [11]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [12]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for inference

In [13]:
current_res = 1.0

In [14]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready/climate_zones_1p0.csv')

In [15]:
zones_df = pandas.read_csv(mlready_data_path)

In [17]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [18]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

In [19]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [ ]:
random_seed = tutorial_config['random_seed']

In [20]:
test_frac = 0.2
val_frac = 0.2

In [21]:
zones_df['test'] = False
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
zones_df['test'][test_df.index] = True

/var/tmp/ipykernel_321047/1662537359.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['test'][test_df.index] = True


In [22]:
zones_df['val'] = False
val_df = zones_df[zones_df['test'] == False].groupby(['period_start','scenario']).sample(frac=(0.2)/(1.0-test_frac), random_state=random_seed)
zones_df['val'][val_df.index] = True

/var/tmp/ipykernel_321047/2859563780.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['val'][val_df.index] = True


In [23]:
train_df = zones_df[((zones_df['val'] == False) & (zones_df['test'] == False ))]

### Next steps or potential follow on material



###  Exmaples of Use


### Data statement
###     References
